# Learning multivariate effect patterns for MASH

This vignette explains how to estimate a mixture-of-covariances prior, fit MASH and interpret the learned sharing patterns.

## Learning goals

After completing this vignette, you will be able to explain what the mixture prior represents, distinguish prior estimation from posterior fitting, run the three-step example, and interpret covariance components without treating them as discrete biological clusters.

## Background and method

MASH models effects across many related conditions using a mixture of multivariate normal distributions. Each covariance component represents a possible pattern of effect sharing: condition-specific, broadly shared, correlated among a subset of conditions or learned from the data. Mixture weights describe how strongly the observed effects support these patterns.

Prior estimation and posterior inference are separate operations. `mixture_prior.ipynb` learns data-driven covariance components from strong signals and combines them with canonical components. `mash_fit.ipynb` fits the mixture to the full analysis set and computes posterior effect estimates, uncertainties and measures of sharing. Reusing a prior is appropriate only when its conditions and effect architecture are compatible with the target analysis.

The example uses small synthetic 34-context inputs. It demonstrates object flow and interpretation, not a biological reference prior.

## Worked example

The analysis has three sequential steps: estimate the prior, fit MASH, then visualize the covariance components. The paths below define the intended protocol-example bundle, but the complete bundle is not currently shipped: only the prior-estimation fixture is available. The commands show the required interface until the missing MASH input and residual-correlation files are added.

| Role | Example path |
|---|---|
| Prior-estimation input | `input/mash/protocol_example.mashr_input.rds` |
| MASH input | `input/mash/protocol_example.EE.mash.rds` |
| Residual correlation matrix | `input/mash/protocol_example.EE.V_simple.rds` |

The matrices must use the same condition order and compatible units. The prior-estimation object contains random, strong and null subsets plus `XtX`; the MASH input contains the effects and standard errors to which the learned prior is applied.

### Step 1. [Estimate the mixture prior](https://statfungen.github.io/xqtl-protocol/mixture_prior.html)

`ed_bovy` estimates data-driven covariance components and assembles them with canonical components.

**Timing**: TBD

In [ ]:
sos run pipeline/mixture_prior.ipynb ed_bovy \
  --output-prefix protocol_example \
  --data input/mash/protocol_example.mashr_input.rds \
  --cwd output/mixture_prior

### Step 2. [Fit MASH](https://statfungen.github.io/xqtl-protocol/mash_fit.html)

`mash` combines the input effects, residual correlation and estimated prior to compute posterior summaries.

**Timing**: TBD

In [ ]:
sos run pipeline/mash_fit.ipynb mash \
  --output-prefix protocol_example_mash \
  --data input/mash/protocol_example.EE.mash.rds \
  --vhat-data input/mash/protocol_example.EE.V_simple.rds \
  --prior-data output/mixture_prior/protocol_example.EE.prior.rds \
  --effect-model EE \
  --compute-posterior \
  --cwd output/mash_fit

### Step 3. Visualize the learned covariance components

`plot_U` draws heatmaps of the covariance matrices in the fitted prior.

**Timing**: TBD

In [ ]:
sos run pipeline/mixture_prior.ipynb plot_U \
  --output-prefix protocol_example_plots \
  --data output/mixture_prior/protocol_example.EE.prior.rds \
  --cwd output/mixture_prior

The chain produces:

- `output/mixture_prior/protocol_example.EE.prior.rds` — data-driven and canonical covariance components with their mixture structure.
- `output/mash_fit/protocol_example_mash.*.rds` — fitted MASH model and, when requested, posterior summaries.
- `output/mixture_prior/protocol_example_plots.EE.prior.pdf` — covariance-component heatmaps.

Run the steps in order. Step 2 uses the prior from Step 1; Step 3 visualizes that same prior.

### Command reference

In [ ]:
sos run pipeline/mixture_prior.ipynb -h
sos run pipeline/mash_fit.ipynb -h

## Results and interpretation

### Mixture-prior result

The prior contains covariance matrices and mixture information. Inspect dimensions, names and weights rather than printing every matrix.

| Result | Interpretation |
|---|---|
| Covariance component | Candidate pattern of effect sharing across conditions |
| Diagonal values | Relative effect variance within conditions |
| Off-diagonal values | Direction and strength of shared effects |
| Mixture weight | Support for a component in the fitted effect distribution |
| Posterior summaries | Shrunk effects and uncertainty after combining data with the mixture |

In [ ]:
prior <- readRDS("output/mixture_prior/protocol_example.EE.prior.rds")
cat("Class:", class(prior), "\n")
cat("Top-level fields:", paste(names(prior), collapse = ", "), "\n")

fit_files <- list.files("output/mash_fit", pattern = "^protocol_example_mash.*\\.rds$", full.names = TRUE)
fit <- readRDS(fit_files[[1]])
cat("MASH result fields:", paste(names(fit), collapse = ", "), "\n")

A covariance heatmap is a pattern used by the mixture model, not a cluster assignment for samples or conditions. Strong positive off-diagonal structure supports effects with similar direction across conditions; near-diagonal structure supports condition-specific effects. Interpret a pattern together with its estimated weight and posterior results. Components can overlap, and several components may approximate the same broad biological pattern.

## Limitations and common pitfalls

- Condition names and ordering must match across all matrices.
- Strong-effect subsets used to learn covariances can bias the prior if they are unrepresentative.
- A prior learned in one study should not be transferred automatically to another tissue set or measurement scale.
- Residual correlation `V` and biological effect covariance are different quantities.
- Similar covariance components can be non-identifiable; interpret broad patterns rather than every component literally.
- The complete protocol-example input bundle is currently missing, so the three-step worked example cannot yet run end to end.
- Synthetic inputs demonstrate mechanics, not expected biological sharing.

## Next steps

Use the fitted posterior summaries to identify effects that are shared or condition-specific, then compare those conclusions with sensitivity analyses using alternative covariance sets or residual-correlation estimates. Record the condition order and prior provenance with every downstream result.